Pupil Preprocessing (Participants P01–P20)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import logging
import importlib
import traceback

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.stats import median_abs_deviation
from pathlib import Path

In [ ]:
PARTICIPANTS =  ["P12"]# [f"P{i:02d}" for i in range(9, 10) if i not in (11, 12, 13)]

BASE_DIR = "/content/drive/MyDrive/CAMES/data_collection_training"

# ── Pupil pipeline parameters ──────────────────────────────────────────────────
RANGE       = (1.0, 10.0)   # valid pupil diameter range in mm
BLINK_PAD_S = 0.10          # seconds to pad each side of a blink
INTERP_S    = 0.20          # max gap to interpolate in seconds
SMOOTH_S    = 0.50          # Savitzky-Golay window in seconds

PUPIL_COLS = [
    "left_pupil_diameter",  "right_pupil_diameter",
    "left_pupil_validity",  "right_pupil_validity",
]

#Logging
importlib.reload(logging)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger(__name__)

In [ ]:
# ── Pupil cleaning function ────────────────────────────────────────────────────
def clean_pupil(df: pd.DataFrame) -> pd.DataFrame:
    """Apply full pupil pipeline to a single-participant gaze DataFrame."""
    df = df.copy()
    df[PUPIL_COLS] = df[PUPIL_COLS].apply(pd.to_numeric, errors="coerce")

    freq = 1.0 / df["time_s"].diff().median()

    # 1. Validity masks
    bad_L = (
        df["left_pupil_diameter"].le(0)  |
        df["left_pupil_diameter"].isna() |
        df["left_pupil_validity"].lt(0.5)
    )
    bad_R = (
        df["right_pupil_diameter"].le(0)  |
        df["right_pupil_diameter"].isna() |
        df["right_pupil_validity"].lt(0.5)
    )

    # 2. Blink mask with padding
    PAD   = int(BLINK_PAD_S * freq)
    blink = (bad_L & bad_R).rolling(2 * PAD + 1, center=True, min_periods=1).max().astype(bool)
    df["blink"] = blink

    # 3. Binocular average → pupil_raw
    with np.errstate(all="ignore"):
        df["pupil_raw"] = np.nanmean(np.vstack([
            df["left_pupil_diameter"].where(~bad_L),
            df["right_pupil_diameter"].where(~bad_R),
        ]), axis=0)
    df["pupil_raw"] = df["pupil_raw"].where(~blink)

    # 4. Spike removal + range clip
    pupil    = df["pupil_raw"].copy()
    roll_med = pupil.rolling(11, center=True, min_periods=1).median()
    mad      = median_abs_deviation(pupil.dropna())
    pupil    = pupil.where((pupil - roll_med).abs() <= 3 * mad)
    pupil    = pupil.where(pupil.between(*RANGE))
    df["pupil_clean"] = pupil

    # 5. Interpolate short gaps
    MAX_GAP = int(INTERP_S * freq)
    df["pupil_interp"] = (
        df["pupil_clean"]
        .interpolate(method="linear", limit=MAX_GAP, limit_direction="both")
    )

    # 6. Savitzky-Golay smoothing
    WIN = max(int(SMOOTH_S * freq) | 1, 5)
    tmp = df["pupil_interp"].interpolate(method="linear", limit_direction="both")
    df["pupil_smooth"] = np.where(
        df["pupil_interp"].notna(),
        savgol_filter(tmp, window_length=WIN, polyorder=2),
        np.nan,
    )

    return df

In [ ]:
# ── QC report ─────────────────────────────────────────────────────────────────
def qc_report(pid: str, df: pd.DataFrame) -> None:
    freq         = 1.0 / df["time_s"].diff().median()
    session_min  = (df["time_s"].max() - df["time_s"].min()) / 60
    blink_onsets = (df["blink"] & ~df["blink"].shift(fill_value=False)).sum()

    log.info("  QC — fs: %dHz  session: %.1fmin  blink: %.1f%%  blinks/min: %.1f  missing_clean: %.1f%%  missing_interp: %.1f%%  pupil_mean: %.2fmm",
        round(freq),
        session_min,
        df["blink"].mean() * 100,
        blink_onsets / session_min,
        df["pupil_clean"].isna().mean() * 100,
        df["pupil_interp"].isna().mean() * 100,
        df["pupil_smooth"].mean() if df["pupil_smooth"].notna().any() else float("nan"),
    )

In [ ]:
# ── Main loop ──────────────────────────────────────────────────────────────────
def main():
    log.info("Processing %d participant(s).", len(PARTICIPANTS))

    for pid in PARTICIPANTS:
        print(f"Starting {pid}...")
        log.info("── %s ──────────────────────────────────────────", pid)

        eye_dir    = os.path.join(BASE_DIR, pid, f"{pid}_session1", f"eye_{pid}")
        gaze_path  = os.path.join(eye_dir, "gaze_marked.csv")
        out_path   = os.path.join(eye_dir, "gaze_with_pupil.csv")

        if not os.path.exists(gaze_path):
            log.error("  gaze_marked.csv not found: %s", gaze_path)
            continue

        try:
            # 1. Load
            df = pd.read_csv(gaze_path, low_memory=False)
            log.info("  Loaded %d samples", len(df))

            # 2. Ensure time_s exists (compute if missing)
            if "time_s" not in df.columns:
                df["device_time_stamp"] = pd.to_numeric(df["device_time_stamp"], errors="coerce")
                df = df.sort_values("Timestamp Unix").reset_index(drop=True)
                df["time_s"] = (
                    df["device_time_stamp"] - df["device_time_stamp"].iloc[0]
                ) / 1e6

            # 3. Run pupil pipeline
            df = clean_pupil(df)

            # 4. QC report
            qc_report(pid, df)

            # 5. Save
            df.to_csv(out_path, index=False)
            log.info("  Saved → %s", out_path)

        except Exception:
            log.error("  FAILED:\n%s", traceback.format_exc())

    log.info("All done.")


main()

11:10:41  INFO      Processing 1 participant(s).
11:10:41  INFO      ── P12 ──────────────────────────────────────────


Starting P12...


11:11:01  INFO        Loaded 850215 samples
/tmp/ipykernel_1131/3006444060.py:28: RuntimeWarning: Mean of empty slice
  df["pupil_raw"] = np.nanmean(np.vstack([
11:11:03  INFO        QC — fs: 90Hz  session: 158.4min  blink: 25.3%  blinks/min: 33.1  missing_clean: 25.3%  missing_interp: 9.8%  pupil_mean: 3.30mm
11:11:41  INFO        Saved → /content/drive/MyDrive/CAMES/data_collection_training/P12/P12_session1/eye_P12/gaze_with_pupil.csv
11:11:41  INFO      All done.


NOTES:

gaze_with_pupil.csv is the output of STEP 3 and serves as the main input for feature extraction. It contains everything that was in gaze_marked.csv, all the original Tobii columns (gaze coordinates, validity scores, raw pupil diameters, timestamps) plus the five task and exploration columns added(task_label, task_begin_unix, t_rel_to_task_s, in_exploration, exploration_id)  with five additional columns appended by the pupil cleaning pipeline:

blink: a boolean column (True/False) indicating whether each sample was detected as a blink. A sample is marked as a blink if both eyes simultaneously have invalid pupil data, extended by 100ms padding on each side to also remove the corrupted signal at the edges of each blink where the eyelid is partially covering the pupil.

pupil_raw: the first cleaned version of the pupil signal. It is the average of the left and right pupil diameters at each sample, using whichever eyes are valid. If only one eye is valid at a given sample that eye's value is used alone. Blink samples are set to NaN. This column still contains spikes and out-of-range values.

pupil_clean: the pupil signal after spike removal and range clipping. Samples that deviate from their local rolling median by more than 3 median absolute deviations are removed as spikes. Samples outside the physically plausible range of 1 to 10 millimetres are also removed. Removed samples are set to NaN. This is the most conservative version of the signal.

pupil_interp: the pupil signal after linear interpolation of short gaps. Any NaN gap in pupil_clean that spans 18 samples or fewer (equivalent to 200ms at 90Hz) is filled by drawing a straight line between the last valid value before the gap and the first valid value after it. Gaps longer than 200ms are left as NaN because filling them would mean making up data over too long a period, which could introduce artefacts.

pupil_smooth: the final cleaned pupil signal, ready for feature extraction. A Savitzky-Golay filter with a 500ms window and polynomial order 2 is applied to pupil_interp to remove remaining high-frequency noise while preserving the slower pupil dilation responses that reflect cognitive load. NaN values are not smoothed, only samples where pupil_interp has a valid value receive a smoothed value. This is the column used in STEP 5 as PUPIL_COL = "pupil_smooth".

The reason STEP 5 reads gaze_with_pupil.csv rather than gaze_marked.csv is that it needs both the cleaned pupil signal for pupil feature extraction and the task and exploration labels for filtering,  both of which are present in this file. STEP 3 reads gaze_marked.csv directly since fixation and saccade detection does not depend on the pupil signal.